In [1]:
import numpy as np

In [2]:
from preprocessing import preprocess 
X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="flatten")


Loading MNIST dataset...
Split completed: Train=54000, Val=6000, Test=10000


In [3]:
class KNN:
    def __init__(self, weights='uniform', k=3, metric='euclidean'):
        self.k = k
        self.weights = weights
        self.metric = metric

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    def _compute_distances(self, X):
        """Compute distances from all test points to all training points at once."""
        if self.metric == 'euclidean':
            # ||a-b||^2 = ||a||^2 + ||b||^2 - 2*a·b
            a2 = np.sum(X ** 2, axis=1, keepdims=True)           # (n_test, 1)
            b2 = np.sum(self.X_train ** 2, axis=1, keepdims=True) # (n_train, 1)
            dists = np.sqrt(np.maximum(a2 + b2.T - 2 * X @ self.X_train.T, 0))

        elif self.metric == 'manhattan':
            # Loop over test points but vectorize over training points
            dists = np.array([
                np.sum(np.abs(self.X_train - x), axis=1) for x in X
            ])

        elif self.metric == 'cosine':
            X_norm = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
            T_norm = self.X_train / (np.linalg.norm(self.X_train, axis=1, keepdims=True) + 1e-9)
            dists = 1 - X_norm @ T_norm.T

        else:
            raise ValueError(f"Metric '{self.metric}' not supported in vectorized mode")

        return dists  # shape: (n_test, n_train)

    def predict(self, X):
        X = np.array(X)
        dists = self._compute_distances(X)          # (n_test, n_train)
        k_idx = np.argsort(dists, axis=1)[:, :self.k]  # (n_test, k)

        predictions = []
        for i, neighbors in enumerate(k_idx):
            labels = self.y_train[neighbors]
            if self.weights == 'uniform':
                weights = np.ones(self.k)
            else:
                weights = 1 / (dists[i, neighbors] + 1e-9)

            votes = {}
            for label, w in zip(labels, weights):
                votes[label] = votes.get(label, 0) + w
            predictions.append(max(votes, key=votes.get))

        return np.array(predictions)

    def score(self, X, y):
        return np.mean(self.predict(X) == y)

In [4]:
# Debug settings (custom KNN is very slow on full MNIST)
#create an instance of the KNN class
knn = KNN(weights='uniform', k=3, metric='euclidean') 
#fit the model to the training data
knn.fit(X_train, y_train) 
# X_val_small = X_val[:100]
# y_val_small = y_val[:100] 
# print(knn.score(X_val_small, y_val_small))


In [5]:
def evaluate(X, y, dataset_name="Validation"):
    y_pred = knn.predict(X)
    y_true = y

    # Positive class = 0 (digit zero)
    TP = np.sum((y_pred == 0) & (y_true == 0))
    TN = np.sum((y_pred == 1) & (y_true == 1))
    FP = np.sum((y_pred == 0) & (y_true == 1))
    FN = np.sum((y_pred == 1) & (y_true == 0))

    accuracy  = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    print(f"--- {dataset_name} Results ---")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-Score  : {f1:.4f}")

    print("Confusion Matrix:")
    print(f"                 Predicted 0   Predicted 1")
    print(f"  Actual 0   :   {TP:<12}  {FN}")
    print(f"  Actual 1   :   {FP:<12}  {TN}")

In [ ]:
evaluate(X_val, y_val, dataset_name="Validation FULL")

In [6]:
evaluate(X_test, y_test, dataset_name="Test")

--- Test Results ---
Accuracy  : 0.9969
Precision : 0.9750
Recall    : 0.9939
F1-Score  : 0.9843
Confusion Matrix:
                 Predicted 0   Predicted 1
  Actual 0   :   974           6
  Actual 1   :   25            8995
